In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [3]:
import os
import pandas as pd
import torch
from ogb.nodeproppred import PygNodePropPredDataset

# 1. Load the graph
dataset = PygNodePropPredDataset(name='ogbn-arxiv')
data = dataset[0]

# 2. Load Mapping Files
# The 'nodeidx2paperid' file maps: Node Index -> MAG Paper ID
path = os.path.join(dataset.root, 'mapping', 'nodeidx2paperid.csv.gz')
nodeidx2paperid = pd.read_csv(path)

# The 'titleabs' file contains: MAG Paper ID -> Title/Abstract
if not os.path.exists('titleabs.tsv'):
    !wget https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
    !gunzip titleabs.tsv.gz

titleabs = pd.read_csv('titleabs.tsv', sep='\t', names=['paper id', 'title', 'abstract'])

nodeidx2paperid['paper id'] = pd.to_numeric(nodeidx2paperid['paper id'], errors='coerce')
titleabs['paper id'] = pd.to_numeric(titleabs['paper id'], errors='coerce')
nodeidx2paperid = nodeidx2paperid.dropna(subset=['paper id']).astype({'paper id': 'int64'})
titleabs = titleabs.dropna(subset=['paper id']).astype({'paper id': 'int64'})

print("Aligning text to node indices...")
# We merge on nodeidx2paperid so the final dataframe order matches the Graph Node Indices (0 to 169,342)
df = pd.merge(nodeidx2paperid, titleabs, on='paper id', how='left')

# Handle missing text (if any papers in the graph aren't in the tsv)
df['title'] = df['title'].fillna('Unknown Title')
df['abstract'] = df['abstract'].fillna('Unknown Abstract')
df['text'] = df['title'] + ". " + df['abstract']

texts = df['text'].tolist()
print(f"Successfully aligned {len(texts)} texts.")

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:01<00:00, 74.20it/s]
Processing...


Extracting dataset/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 9098.27it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 372.53it/s]

Saving...



Done!
/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


--2026-04-20 16:06:35--  https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 70213527 (67M) [application/x-gzip]
Saving to: ‘titleabs.tsv.gz’

titleabs.tsv.gz     100%[===================>]  66.96M  36.3MB/s    in 1.8s    

2026-04-20 16:06:37 (36.3 MB/s) - ‘titleabs.tsv.gz’ saved [70213527/70213527]

Aligning text to node indices...
Successfully aligned 169343 texts.


In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prepare Graph
data.edge_index = to_undirected(data.edge_index)
data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)
data = data.to(device)

split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
valid_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)

/tmp/ipykernel_55/3115342646.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)


In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

# class GraphSAGE(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
#         super(GraphSAGE, self).__init__()
#         self.convs = torch.nn.ModuleList()
#         self.bns = torch.nn.ModuleList()
        
#         # Input Layer
#         # normalize=True adds the L2 normalization often used in GraphSAGE papers
#         self.convs.append(SAGEConv(in_channels, hidden_channels, normalize=True))
#         self.bns.append(nn.BatchNorm1d(hidden_channels))
        
#         # Hidden Layers
#         for _ in range(num_layers - 2):
#             self.convs.append(SAGEConv(hidden_channels, hidden_channels, normalize=True))
#             self.bns.append(nn.BatchNorm1d(hidden_channels))
            
#         # Output Layer
#         self.convs.append(SAGEConv(hidden_channels, out_channels, normalize=True))
#         self.dropout = dropout

#     def forward(self, x, edge_index):
#         for i in range(len(self.convs) - 1):
#             # Save for residual connection
#             identity = x if i > 0 and x.size(-1) == self.convs[i].out_channels else None
#             x = self.convs[i](x, edge_index)
#             x = self.bns[i](x)
#             x = F.relu(x)
            
#             if identity is not None:
#                 x = x + identity
            
#             x = F.dropout(x, p=self.dropout, training=self.training)
            
#         x = self.convs[-1](x, edge_index)
#         return x

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
        super(GraphSAGE, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        
        # Input Layer
        self.convs.append(SAGEConv(in_channels, hidden_channels, normalize=True))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Hidden Layers
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, normalize=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Final Fusion Layer (Jumping Knowledge)
        # We concatenate all layers, so the input is hidden_channels * num_layers
        self.final_lin = nn.Linear(hidden_channels * num_layers, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        layer_outputs = []
        
        for i in range(len(self.convs)):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)
            
        # Jumping Knowledge: Concatenate all layer outputs
        x = torch.cat(layer_outputs, dim=-1)
        x = self.final_lin(x)
        
        return x
        

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = torch.nn.CrossEntropyLoss()

In [6]:
import numpy as np

# Multi-Run Execution
all_results = []
for run in range(10):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/10 (Seed: {seed})")
    
    model = GraphSAGE(
        in_channels=768,
        hidden_channels=256, 
        out_channels=dataset.num_classes,
        num_layers=4,
        dropout=0.3
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    best_val, best_test = 0, 0
    for epoch in range(1, 501):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                preds = logits.argmax(dim=-1, keepdim=True)
                val_acc = (preds[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (preds[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)
                
                if val_acc > best_val:
                    best_val, best_test = val_acc, test_acc
            print(f"Run {run+1} | Epoch {epoch} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")
            
    all_results.append(best_test)
    print(f"Run {run+1} Finished. Best Test Acc: {best_test:.4f}")

print(f"\nFinal Test Accuracy: {np.mean(all_results):.4f} ± {np.std(all_results):.4f}")


>>> Starting Run 1/10 (Seed: 0)
Run 1 | Epoch 20 | Val: 0.4890 | Test: 0.5147
Run 1 | Epoch 40 | Val: 0.6576 | Test: 0.6658
Run 1 | Epoch 60 | Val: 0.6928 | Test: 0.6837
Run 1 | Epoch 80 | Val: 0.7359 | Test: 0.7283
Run 1 | Epoch 100 | Val: 0.7343 | Test: 0.7244
Run 1 | Epoch 120 | Val: 0.7459 | Test: 0.7322
Run 1 | Epoch 140 | Val: 0.7348 | Test: 0.7360
Run 1 | Epoch 160 | Val: 0.7372 | Test: 0.7196
Run 1 | Epoch 180 | Val: 0.7345 | Test: 0.7236
Run 1 | Epoch 200 | Val: 0.7220 | Test: 0.6995
Run 1 | Epoch 220 | Val: 0.7188 | Test: 0.7003
Run 1 | Epoch 240 | Val: 0.7294 | Test: 0.7243
Run 1 | Epoch 260 | Val: 0.7323 | Test: 0.7098
Run 1 | Epoch 280 | Val: 0.7444 | Test: 0.7299
Run 1 | Epoch 300 | Val: 0.7300 | Test: 0.7164
Run 1 | Epoch 320 | Val: 0.7164 | Test: 0.6884
Run 1 | Epoch 340 | Val: 0.7369 | Test: 0.7292
Run 1 | Epoch 360 | Val: 0.7400 | Test: 0.7301
Run 1 | Epoch 380 | Val: 0.7053 | Test: 0.6729
Run 1 | Epoch 400 | Val: 0.7324 | Test: 0.7214
Run 1 | Epoch 420 | Val: 0.7286